# ЛР № 4. Витрина почасового спроса Seoul Bike Sharing Demand

**Выбранный вопрос - основной вопрос [Project Brief](../docs/project_brief.md):** как различается средний
почасовой спрос на прокат велосипедов по часам суток в разные сезоны и в рабочие, выходные и праздничные дни?

Метрики из Project Brief: средний и медианный почасовой спрос - `avg` и `median` числа аренд по часам,
когда прокат работал; число учтённых часов группы; суммарное число аренд и доля в итоге за период.
Срезы: час суток, сезон и тип дня (рабочий день, выходной, праздник). Уточняющие вопросы о погоде
в этой витрине не рассматриваются.

Вход - `staging`, подготовленный в ЛР № 3 ([отчёт](../docs/data_quality_report.md)).
Модель и происхождение полей описаны в [docs/mart.md](../docs/mart.md). Полный расчёт метрик
и содержательные выводы остаются следующему этапу.

## 1. Параметры и вход

Логическая дата берётся из пути `staging`, а не из даты запуска. `source_ingested_on` обозначает загрузку
исходного файла и не используется как время наблюдения: наблюдения относятся к периоду
с 2017-12-01 по 2018-11-30.

Ноутбук запускается из корня репозитория или каталога `notebooks` с ядром проектного `.venv`.
Стенд должен работать, настройки S3 читаются из `.env`. Ingestion и ЛР № 3 повторно выполнять не нужно.

In [1]:
from datetime import date

DATASET_SLUG = "seoul_bike_sharing"
SOURCE_INGESTED_ON = "2026-10-24"
MART_NAME = "hourly_rentals"
date.fromisoformat(SOURCE_INGESTED_ON)

PREFIX = f"{DATASET_SLUG}/source_ingested_on={SOURCE_INGESTED_ON}"
STAGING_KEY = f"{PREFIX}/staging.parquet"
STAGING_PATH = f"s3://staging/{STAGING_KEY}"
MART_KEY = f"{PREFIX}/{MART_NAME}.parquet"
MART_PATH = f"s3://mart/{MART_KEY}"

print(f"Вход:  {STAGING_PATH}")
print(f"Выход: {MART_PATH}")

Вход:  s3://staging/seoul_bike_sharing/source_ingested_on=2026-10-24/staging.parquet
Выход: s3://mart/seoul_bike_sharing/source_ingested_on=2026-10-24/hourly_rentals.parquet


In [2]:
import os
import sys
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file() and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
required_settings = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise RuntimeError(f"Не заданы настройки: {', '.join(missing_settings)}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_s3  # noqa: E402


def literal(value):
    return "'" + value.replace("'", "''") + "'"


def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


check_results = []


def check(label, actual, expected):
    passed = actual == expected
    check_results.append((label, actual, expected, passed))
    assert passed, f"{label}: получено {actual!r}, ожидалось {expected!r}"


connection = duckdb.connect()
configure_s3(connection)
endpoint = os.environ["S3_ENDPOINT"]
client = boto3.client(
    "s3",
    endpoint_url=endpoint if endpoint.startswith("http") else f"http://{endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)
source_before = client.head_object(Bucket="staging", Key=STAGING_KEY)

Читаем `staging` напрямую из S3, проверяем наличие нужных полей и повторно убеждаемся, что на входе нет
нарушений критических правил ЛР № 3. Ограничения качества, влияющие на будущие показатели:

- 295 часов с `is_functioning_hour = false` - прокат не работал, число аренд равно нулю. Они остаются
  в витрине, но не участвуют в среднем и медианном спросе;
- флаг ЛР № 3 `quality_humidity_in_physical_range_violated` (17 часов) относится только к влажности,
  которая не нужна для выбранного вопроса; эти часы участвуют в метриках спроса без ограничений;
- признак праздника определён источником и расходится с официальным календарём (Project Brief, раздел 5.1).

In [3]:
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW staging AS
    SELECT * FROM read_parquet({literal(STAGING_PATH)}, hive_partitioning = false)
""")
source_schema = {row[0]: row[1] for row in connection.execute("DESCRIBE staging").fetchall()}
required_fields = {
    "rental_date", "rental_hour", "rented_bike_count", "season", "is_holiday",
    "is_functioning_hour", "quality_humidity_in_physical_range_violated",
}
check("Наличие нужных полей staging", required_fields - source_schema.keys(), set())

source_counts = connection.execute("""
    SELECT
        count(*) AS rows_count,
        count(*) FILTER (WHERE is_functioning_hour) AS functioning_hours,
        count(*) FILTER (WHERE NOT is_functioning_hour) AS non_functioning_hours,
        count(*) FILTER (WHERE quality_humidity_in_physical_range_violated) AS humidity_flags,
        count(*) FILTER (
            WHERE rental_date IS NULL OR rental_hour IS NULL OR rented_bike_count IS NULL
               OR season IS NULL OR is_holiday IS NULL OR is_functioning_hour IS NULL
               OR rental_hour NOT BETWEEN 0 AND 23 OR rented_bike_count < 0
               OR (NOT is_functioning_hour AND rented_bike_count <> 0)
        ) AS critical_violations
    FROM staging
""").fetchone()
(staging_rows, source_functioning, source_non_functioning, source_humidity_flags,
 source_critical) = source_counts
check("Критические нарушения ЛР № 3 на входе", source_critical, 0)
show_table(
    ["Строк staging", "Прокат работал", "Прокат не работал", "Флагов влажности",
     "Критических нарушений"],
    [source_counts],
)
show_table(["Нужное поле staging", "Тип"],
           [(name, source_schema[name]) for name in sorted(required_fields)])

source_measure_profile = connection.execute("""
    SELECT count(*) FILTER (WHERE rented_bike_count IS NULL),
           min(rented_bike_count), max(rented_bike_count), sum(rented_bike_count)
    FROM staging
""").fetchone()
show_table(["NULL аренд", "Минимум", "Максимум", "Сумма аренд"], [source_measure_profile])

Строк staging,Прокат работал,Прокат не работал,Флагов влажности,Критических нарушений
8760,8465,295,17,0


Нужное поле staging,Тип
is_functioning_hour,BOOLEAN
is_holiday,BOOLEAN
quality_humidity_in_physical_range_violated,BOOLEAN
rental_date,DATE
rental_hour,INTEGER
rented_bike_count,INTEGER
season,VARCHAR


NULL аренд,Минимум,Максимум,Сумма аренд
0,0,3556,6172314


## 2. Гранулярность, ключ и меры

**Одна строка `hourly_rentals` представляет один час наблюдения городской системы проката в одну
календарную дату выбранного снимка - и часы, когда прокат работал, и часы его остановки.**

Ключ - естественный составной ключ `rental_date` × `rental_hour`. В ЛР № 3 правило `date_hour_unique`
подтвердило, что каждая пара встречается один раз, а ряд полный: 365 дат по 24 часа. Технический ключ
не создаётся, `DISTINCT` не применяется. Исходные данные уже агрегированы по часу: `COUNT(*)` в витрине
считает часы наблюдения, а число аренд хранится в отдельной мере `rented_bike_count`. Отдельные аренды
из часовых итогов не восстанавливаются.

| Мера или признак | Единица | Использование в вопросе | Правило агрегации |
| --- | --- | --- | --- |
| `rented_bike_count` | Аренды за час | Средний и медианный почасовой спрос, суммарное число аренд | Сумма допустима по непересекающимся часам одного снимка и по любым срезам. Среднее и медиана считаются только по часам с `is_functioning_hour = true`; средние и медианы групп не складываются и не усредняются без весов, а пересчитываются из часов нужной группы |
| `is_functioning_hour` | Признак | Отбор часов для среднего и медианы; число нерабочих часов группы | Число часов с признаком складывается по непересекающимся группам |
| `count(*)` | Часы | Число учтённых часов группы и правило малых групп (меньше 10 часов) | Складывается по непересекающимся группам |

Временные поля: `rental_date` и `rental_hour` описывают период наблюдения - час события. Логическая
дата загрузки `source_ingested_on` остаётся только в пути и не заменяет время наблюдения; дата сбора
исходного снимка источником не указана.

Структура - одна широкая таблица. Все срезы основного вопроса являются атрибутами самой строки или
выводятся из её даты: сезон, признак праздника, ISO-день недели и тип дня. Отдельное измерение дат
(365 строк) потребовало бы `JOIN` без новой информации, поэтому не создаётся. `JOIN` в модели нет.

## 3. SQL-преобразование

Поля выбраны для основного вопроса:

- переносятся без изменения: `rental_date`, `rental_hour`, `season`, `is_holiday`,
  `is_functioning_hour`, `rented_bike_count`;
- `rental_isodow` - ISO-день недели даты: 1 - понедельник, 7 - воскресенье;
- `day_type` - тип дня по правилу Project Brief: `holiday`, если `is_holiday`; иначе `weekend` для
  субботы и воскресенья; иначе `workday`. Праздник имеет приоритет: в снимке четыре праздничные даты
  выпали на выходные.

Погодные поля и флаг влажности ЛР № 3 к вопросу не относятся и в витрину не входят. Фильтрации строк нет:
нерабочие часы сохраняются, иначе изменилось бы число часов наблюдения и потерялась бы информация
о нерабочих часах групп. Баланс: `staging = витрина + исключённые строки`, исключённых строк 0.

In [4]:
connection.execute("""
    CREATE OR REPLACE TEMP TABLE hourly_rentals AS
    SELECT
        rental_date,
        rental_hour,
        CAST(isodow(rental_date) AS INTEGER) AS rental_isodow,
        CASE
            WHEN is_holiday THEN 'holiday'
            WHEN isodow(rental_date) IN (6, 7) THEN 'weekend'
            ELSE 'workday'
        END AS day_type,
        season,
        is_holiday,
        is_functioning_hour,
        rented_bike_count
    FROM staging
    ORDER BY rental_date, rental_hour
""")
prepared_rows = connection.execute("SELECT count(*) FROM hourly_rentals").fetchone()[0]
excluded_rows = staging_rows - prepared_rows
check("Исключённые строки при сборке", excluded_rows, 0)
check("Баланс staging = витрина + исключено", staging_rows, prepared_rows + excluded_rows)
show_table(["Строк staging", "Строк витрины", "Исключено"],
           [(staging_rows, prepared_rows, excluded_rows)])
show_table(
    ["Дата", "Час", "ISO-день", "Тип дня", "Сезон", "Праздник", "Прокат работал", "Аренд"],
    connection.execute(
        "SELECT * FROM hourly_rentals ORDER BY rental_date, rental_hour LIMIT 5"
    ).fetchall(),
)

Строк staging,Строк витрины,Исключено
8760,8760,0


Дата,Час,ISO-день,Тип дня,Сезон,Праздник,Прокат работал,Аренд
2017-12-01,0,5,workday,Winter,False,True,254
2017-12-01,1,5,workday,Winter,False,True,204
2017-12-01,2,5,workday,Winter,False,True,173
2017-12-01,3,5,workday,Winter,False,True,107
2017-12-01,4,5,workday,Winter,False,True,78


## 4. Запись и повторное чтение S3

Используем способ ЛР № 3: удаляем только целевой объект `hourly_rentals.parquet` и записываем его заново.
Повторный запуск заменяет результат, а не добавляет строки. Остальные объекты S3 не затрагиваются.

После записи открываем новое соединение DuckDB и читаем именно объект в S3. Автоматическое добавление
полей из каталогов отключено: проверяется физическая схема файла.

In [5]:
assert MART_PATH == f"s3://mart/{PREFIX}/{MART_NAME}.parquet"
client.head_bucket(Bucket="mart")
client.delete_object(Bucket="mart", Key=MART_KEY)
connection.execute(f"COPY hourly_rentals TO {literal(MART_PATH)} (FORMAT PARQUET)")

published = duckdb.connect()
configure_s3(published)
published.execute(f"""
    CREATE OR REPLACE TEMP VIEW published_rentals AS
    SELECT * FROM read_parquet({literal(MART_PATH)}, hive_partitioning = false)
""")
output_head = client.head_object(Bucket="mart", Key=MART_KEY)
show_table(["Объект", "Размер, байт"], [(MART_PATH, output_head["ContentLength"])])

Объект,"Размер, байт"
s3://mart/seoul_bike_sharing/source_ingested_on=2026-10-24/hourly_rentals.parquet,33045


<a id="checks"></a>
## 5. Проверки сохранённой витрины

Все запросы к `published_rentals` ниже читают записанный Parquet-файл в S3. Ожидания следуют из модели
и входного `staging`. При нарушении любого условия `assert` останавливает выполнение; сводка в конце
показывает фактическое и ожидаемое значение каждой проверки.

### 5.1. Количество и схема

Все строки `staging` сохраняются: проверяем баланс, совпадение с подготовленной таблицей, имена, порядок
и типы восьми полей.

In [6]:
mart_rows = published.execute("SELECT count(*) FROM published_rentals").fetchone()[0]
check("Строк после чтения S3 = строк staging", mart_rows, staging_rows)
check("Строк после чтения S3 = подготовлено", mart_rows, prepared_rows)

expected_schema = [
    ("rental_date", "DATE"),
    ("rental_hour", "INTEGER"),
    ("rental_isodow", "INTEGER"),
    ("day_type", "VARCHAR"),
    ("season", "VARCHAR"),
    ("is_holiday", "BOOLEAN"),
    ("is_functioning_hour", "BOOLEAN"),
    ("rented_bike_count", "INTEGER"),
]
actual_schema = [row[:2] for row in published.execute("DESCRIBE published_rentals").fetchall()]
check("Имена, порядок и типы полей", actual_schema, expected_schema)
show_table(["Staging", "Подготовлено", "Прочитано из mart", "Исключено"],
           [(staging_rows, prepared_rows, mart_rows, excluded_rows)])
show_table(["Поле", "Тип"], actual_schema)

Staging,Подготовлено,Прочитано из mart,Исключено
8760,8760,8760,0


Поле,Тип
rental_date,DATE
rental_hour,INTEGER
rental_isodow,INTEGER
day_type,VARCHAR
season,VARCHAR
is_holiday,BOOLEAN
is_functioning_hour,BOOLEAN
rented_bike_count,INTEGER


### 5.2. Ключ и обязательные поля

Отдельно проверяем пропуски в каждом компоненте ключа и повторы пары `rental_date` × `rental_hour`,
а также полноту ряда. Все остальные поля обязательны: без них теряется смысл строки или нужный срез.

In [7]:
null_keys = published.execute("""
    SELECT count(*) FILTER (WHERE rental_date IS NULL), count(*) FILTER (WHERE rental_hour IS NULL)
    FROM published_rentals
""").fetchone()
duplicate_keys = published.execute("""
    SELECT rental_date, rental_hour, count(*) AS rows_per_key
    FROM published_rentals
    WHERE rental_date IS NOT NULL AND rental_hour IS NOT NULL
    GROUP BY rental_date, rental_hour
    HAVING count(*) > 1
""").fetchall()
coverage = published.execute("""
    SELECT count(DISTINCT rental_date), min(rental_date), max(rental_date),
           min(hours), max(hours)
    FROM (SELECT rental_date, count(*) AS hours FROM published_rentals GROUP BY rental_date)
""").fetchone()
check("NULL в rental_date", null_keys[0], 0)
check("NULL в rental_hour", null_keys[1], 0)
check("Повторяющиеся ключи", len(duplicate_keys), 0)
check("Часов на каждую дату", coverage[3:], (24, 24))

other_nulls = published.execute("""
    SELECT
        count(*) FILTER (WHERE rental_isodow IS NULL),
        count(*) FILTER (WHERE day_type IS NULL),
        count(*) FILTER (WHERE season IS NULL),
        count(*) FILTER (WHERE is_holiday IS NULL),
        count(*) FILTER (WHERE is_functioning_hour IS NULL)
    FROM published_rentals
""").fetchone()
other_fields = ("rental_isodow", "day_type", "season", "is_holiday", "is_functioning_hour")
for name, value in zip(other_fields, other_nulls, strict=True):
    check(f"NULL в обязательном поле {name}", value, 0)
show_table(["NULL rental_date", "NULL rental_hour", "Повторяющихся ключей"],
           [(*null_keys, len(duplicate_keys))])
show_table(["Дат", "Первая дата", "Последняя дата", "Мин. часов на дату", "Макс. часов на дату"],
           [coverage])
show_table(["Поле", "NULL"], zip(other_fields, other_nulls))

NULL rental_date,NULL rental_hour,Повторяющихся ключей
0,0,0


Дат,Первая дата,Последняя дата,Мин. часов на дату,Макс. часов на дату
365,2017-12-01,2018-11-30,24,24


Поле,NULL
rental_isodow,0
day_type,0
season,0
is_holiday,0
is_functioning_hour,0


### 5.3. Меры и вычисленные поля

`rented_bike_count` должна быть заполнена в каждой строке, как и в `staging`. Минимум, максимум и сумма
должны совпасть со `staging`; в нерабочие часы аренд нет, в рабочие - больше нуля, как установлено
в ЛР № 3. Для вычисленных полей проверяем диапазоны и пересчитываем их из даты: час 0-23, ISO-день 1-7,
тип дня - одно из трёх значений по правилу модели.

In [8]:
measure_profile = published.execute("""
    SELECT count(*) FILTER (WHERE rented_bike_count IS NULL),
           min(rented_bike_count), max(rented_bike_count), sum(rented_bike_count)
    FROM published_rentals
""").fetchone()
check("NULL, минимум, максимум и сумма аренд совпадают со staging",
      measure_profile, source_measure_profile)
functioning_profile = published.execute("""
    SELECT
        count(*) FILTER (WHERE NOT is_functioning_hour AND rented_bike_count <> 0),
        min(rented_bike_count) FILTER (WHERE is_functioning_hour)
    FROM published_rentals
""").fetchone()
check("Аренды в нерабочие часы", functioning_profile[0], 0)
check("Минимум аренд в рабочий час больше нуля", functioning_profile[1] > 0, True)

derived_violations = published.execute("""
    SELECT
        count(*) FILTER (WHERE rental_hour NOT BETWEEN 0 AND 23),
        count(*) FILTER (WHERE rental_isodow NOT BETWEEN 1 AND 7),
        count(*) FILTER (WHERE rental_isodow IS DISTINCT FROM isodow(rental_date)),
        count(*) FILTER (WHERE day_type NOT IN ('workday', 'weekend', 'holiday')),
        count(*) FILTER (
            WHERE day_type IS DISTINCT FROM CASE
                WHEN is_holiday THEN 'holiday'
                WHEN rental_isodow IN (6, 7) THEN 'weekend'
                ELSE 'workday'
            END
        )
    FROM published_rentals
""").fetchone()
derived_checks = (
    "Час вне 0-23", "ISO-день вне 1-7", "ISO-день не совпадает с датой",
    "Тип дня вне справочника", "Тип дня не совпадает с правилом",
)
for name, value in zip(derived_checks, derived_violations, strict=True):
    check(name, value, 0)
show_table(["Мера", "NULL", "Минимум", "Максимум", "Сумма"],
           [("rented_bike_count", *measure_profile)])
show_table(["Аренд в нерабочие часы", "Минимум аренд в рабочий час"], [functioning_profile])
show_table(["Проверка вычисленного поля", "Нарушений"], zip(derived_checks, derived_violations))
show_table(["Тип дня", "Дат", "Часов"], published.execute("""
    SELECT day_type, count(DISTINCT rental_date), count(*)
    FROM published_rentals GROUP BY day_type ORDER BY day_type
""").fetchall())

Мера,NULL,Минимум,Максимум,Сумма
rented_bike_count,0,0,3556,6172314


Аренд в нерабочие часы,Минимум аренд в рабочий час
0,2


Проверка вычисленного поля,Нарушений
Час вне 0-23,0
ISO-день вне 1-7,0
ISO-день не совпадает с датой,0
Тип дня вне справочника,0
Тип дня не совпадает с правилом,0


Тип дня,Дат,Часов
holiday,18,432
weekend,100,2400
workday,247,5928


### 5.4. Флаги и преобразования

`is_functioning_hour` - признак источника, а не флаг качества, но именно он определяет совокупность
для среднего спроса, поэтому сверяем число рабочих и нерабочих часов со `staging`.

Флаг ЛР № 3 `quality_humidity_in_physical_range_violated` относится к влажности, которой нет в витрине.
Проверяем, что все 17 помеченных часов присутствуют в витрине и их значения совпадают со `staging`:
решение ЛР № 3 оставить эти часы в данных не изменено.

Затем сверяем все поля записанной витрины с ожидаемой проекцией всех строк `staging` через `EXCEPT ALL`
в обе стороны: значения не должны теряться или искажаться.

In [9]:
functioning_counts = published.execute("""
    SELECT count(*) FILTER (WHERE is_functioning_hour),
           count(*) FILTER (WHERE NOT is_functioning_hour)
    FROM published_rentals
""").fetchone()
check("Рабочие и нерабочие часы совпадают со staging", functioning_counts,
      (source_functioning, source_non_functioning))

published.execute(f"""
    CREATE OR REPLACE TEMP VIEW source_staging AS
    SELECT * FROM read_parquet({literal(STAGING_PATH)}, hive_partitioning = false)
""")
flagged_profile = published.execute("""
    SELECT
        count(*) AS flagged_in_staging,
        count(m.rental_date) AS found_in_mart,
        count(*) FILTER (
            WHERE m.rented_bike_count IS DISTINCT FROM s.rented_bike_count
               OR m.is_functioning_hour IS DISTINCT FROM s.is_functioning_hour
               OR m.season IS DISTINCT FROM s.season
               OR m.is_holiday IS DISTINCT FROM s.is_holiday
        ) AS changed_values
    FROM source_staging AS s
    LEFT JOIN published_rentals AS m
        ON m.rental_date = s.rental_date AND m.rental_hour = s.rental_hour
    WHERE s.quality_humidity_in_physical_range_violated
""").fetchone()
check("Помеченные в ЛР № 3 часы", flagged_profile[0], source_humidity_flags)
check("Помеченные часы найдены в витрине", flagged_profile[1], source_humidity_flags)
check("Значения помеченных часов не изменены", flagged_profile[2], 0)

published.execute("""
    CREATE OR REPLACE TEMP VIEW expected_rentals AS
    SELECT
        rental_date,
        rental_hour,
        CAST(isodow(rental_date) AS INTEGER) AS rental_isodow,
        CASE
            WHEN is_holiday THEN 'holiday'
            WHEN isodow(rental_date) IN (6, 7) THEN 'weekend'
            ELSE 'workday'
        END AS day_type,
        season,
        is_holiday,
        is_functioning_hour,
        rented_bike_count
    FROM source_staging
""")
content_difference = published.execute("""
    SELECT 'Отсутствуют в mart' AS direction, count(*) AS differing_rows
    FROM (SELECT * FROM expected_rentals EXCEPT ALL SELECT * FROM published_rentals)
    UNION ALL
    SELECT 'Лишние в mart', count(*)
    FROM (SELECT * FROM published_rentals EXCEPT ALL SELECT * FROM expected_rentals)
""").fetchall()
for direction, differing_rows in content_difference:
    check(direction, differing_rows, 0)
show_table(["Рабочих часов", "Нерабочих часов"], [functioning_counts])
show_table(["Помечено в staging", "Найдено в mart", "Изменённых значений"], [flagged_profile])
show_table(["Направление сверки", "Различающихся строк"], content_difference)

Рабочих часов,Нерабочих часов
8465,295


Помечено в staging,Найдено в mart,Изменённых значений
17,17,0


Направление сверки,Различающихся строк
Отсутствуют в mart,0
Лишние в mart,0


### 5.5. Пригодность для выбранного вопроса

Контрольный запрос группирует витрину по сезону и типу дня и считает часы наблюдения, рабочие часы, даты
и наименьшее число рабочих часов среди 24 часов суток группы. Сумма часов по группам должна равняться числу
строк витрины. Запрос подтверждает, что все 12 сочетаний срезов основного вопроса представлены, и показывает,
сколько наблюдений получит каждая ячейка «сезон × тип дня × час».

В праздничных группах внутри сезона от 2 до 8 рабочих часов на ячейку - меньше порога 10 из Project Brief.
Поэтому праздничный профиль сравнивается по часам за весь период. Средние, медианы и выводы здесь
не рассчитываются.

In [10]:
slice_counts = published.execute("""
    WITH by_hour AS (
        SELECT season, day_type, rental_hour,
               count(*) AS hours,
               count(*) FILTER (WHERE is_functioning_hour) AS functioning_hours,
               count(DISTINCT rental_date) AS dates
        FROM published_rentals
        GROUP BY season, day_type, rental_hour
    )
    SELECT season, day_type, sum(hours), sum(functioning_hours), max(dates),
           min(functioning_hours)
    FROM by_hour
    GROUP BY season, day_type
    ORDER BY CASE season WHEN 'Winter' THEN 1 WHEN 'Spring' THEN 2 WHEN 'Summer' THEN 3 ELSE 4 END,
             CASE day_type WHEN 'workday' THEN 1 WHEN 'weekend' THEN 2 ELSE 3 END
""").fetchall()
check("Сочетаний сезон × тип дня", len(slice_counts), 12)
check("Баланс контрольной группировки", sum(row[2] for row in slice_counts), mart_rows)
show_table(
    ["Сезон", "Тип дня", "Часов", "Рабочих часов", "Дат",
     "Мин. рабочих часов в ячейке «час суток»"],
    slice_counts,
)
holiday_by_hour = published.execute("""
    SELECT min(functioning_hours), max(functioning_hours)
    FROM (
        SELECT rental_hour, count(*) FILTER (WHERE is_functioning_hour) AS functioning_hours
        FROM published_rentals
        WHERE day_type = 'holiday'
        GROUP BY rental_hour
    )
""").fetchone()
check("Праздничный профиль за период: не меньше 10 рабочих часов в каждом часе суток",
      holiday_by_hour[0] >= 10, True)
show_table(["Праздники за весь период: мин. рабочих часов в часе суток", "Макс."],
           [holiday_by_hour])

Сезон,Тип дня,Часов,Рабочих часов,Дат,Мин. рабочих часов в ячейке «час суток»
Winter,workday,1416,1416,59,59
Winter,weekend,552,552,23,23
Winter,holiday,192,192,8,8
Spring,workday,1512,1464,63,61
Spring,weekend,624,624,26,26
Spring,holiday,72,72,3,3
Summer,workday,1536,1536,64,64
Summer,weekend,624,624,26,26
Summer,holiday,48,48,2,2
Autumn,workday,1464,1296,61,54


Праздники за весь период: мин. рабочих часов в часе суток,Макс.
17,17


### Контрольная сводка

Дополнительно проверяем, что входной объект `staging` не изменился за время выполнения.
Все проверки относятся к конкретному снимку и не доказывают достоверность сведений о реальном спросе.

In [11]:
source_after = client.head_object(Bucket="staging", Key=STAGING_KEY)
check("Входной объект staging не изменён", (
    source_after["ETag"], source_after["ContentLength"], source_after["LastModified"]
), (
    source_before["ETag"], source_before["ContentLength"], source_before["LastModified"]
))
show_table(["Проверка", "Получено", "Ожидается", "Итог"], [
    (name, actual, expected, "Пройдена" if passed else "Нарушение")
    for name, actual, expected, passed in check_results
])
checks_passed = sum(passed for _, _, _, passed in check_results)
show_table(["Проверок", "Пройдено", "Строк mart", "Полей", "Рабочих часов", "Нерабочих часов"], [
    (len(check_results), checks_passed, mart_rows, len(actual_schema), *functioning_counts)
])
published.close()
connection.close()

Проверка,Получено,Ожидается,Итог
Наличие нужных полей staging,set(),set(),Пройдена
Критические нарушения ЛР № 3 на входе,0,0,Пройдена
Исключённые строки при сборке,0,0,Пройдена
Баланс staging = витрина + исключено,8760,8760,Пройдена
Строк после чтения S3 = строк staging,8760,8760,Пройдена
Строк после чтения S3 = подготовлено,8760,8760,Пройдена
"Имена, порядок и типы полей","[('rental_date', 'DATE'), ('rental_hour', 'INTEGER'), ('rental_isodow', 'INTEGER'), ('day_type', 'VARCHAR'), ('season', 'VARCHAR'), ('is_holiday', 'BOOLEAN'), ('is_functioning_hour', 'BOOLEAN'), ('rented_bike_count', 'INTEGER')]","[('rental_date', 'DATE'), ('rental_hour', 'INTEGER'), ('rental_isodow', 'INTEGER'), ('day_type', 'VARCHAR'), ('season', 'VARCHAR'), ('is_holiday', 'BOOLEAN'), ('is_functioning_hour', 'BOOLEAN'), ('rented_bike_count', 'INTEGER')]",Пройдена
NULL в rental_date,0,0,Пройдена
NULL в rental_hour,0,0,Пройдена
Повторяющиеся ключи,0,0,Пройдена


Проверок,Пройдено,Строк mart,Полей,Рабочих часов,Нерабочих часов
34,34,8760,8,8465,295


<a id="result"></a>
## 6. Итог

Витрина сохранена и прочитана из S3:

```text
s3://mart/seoul_bike_sharing/source_ingested_on=2026-10-24/hourly_rentals.parquet
```

Все 8 760 строк `staging` сохранены, исключённых строк нет: гранулярность «дата × час» и число строк
не изменились. Ключ `rental_date` × `rental_hour` заполнен и уникален, у каждой из 365 дат 24 часа.
В файле 8 полей; меры и признаки перенесены без изменения, `rental_isodow` и `day_type` пересчитаны
и проверены. 295 нерабочих часов сохранены с `is_functioning_hour = false` и не участвуют в будущем
среднем спросе. Решения ЛР № 3 не изменены. `JOIN` в модели нет.

Ограничения: один год наблюдений, признак праздника определён источником, а праздничные ячейки
«сезон × тип дня × час» малы, поэтому праздничный профиль сравнивается за весь период. Подробные проверки
находятся в [разделе 5](#checks), описание модели - в [docs/mart.md](../docs/mart.md).